In [1]:
from moe_reft import sft_dataset
from transformers import AutoTokenizer


tokenizer_model_name = "allenai/OLMoE-1B-7B-0125-Instruct"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_model_name)
response_template = sft_dataset.extract_response_template(tokenizer)
response_template_ids = tokenizer(response_template)["input_ids"]
message_extractor = sft_dataset.KeyBasedMessageExtractor(
        user_key="question",
        assistant_key="answer",
        system_message="You are a helpful math tutor. Solve step by step.",
    )

train_dataset = sft_dataset.SFTDataset(
    # source="meta-math/MetaMathQA",
    source="openai/gsm8k",
    tokenizer=tokenizer,
    response_template_ids=response_template_ids,
    message_extractor=message_extractor,
    split="train",
    name="main",
)
val_dataset = sft_dataset.SFTDataset(
    source="openai/gsm8k",
    tokenizer=tokenizer,
    response_template_ids=response_template_ids,
    message_extractor=message_extractor,
    split="test",
    name="main",
)

/home/recoverx/astarag/MoE-ReFT/.venv/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
2026-01-07 13:34:45.442 | INFO     | moe_reft.sft_dataset:validate_one_sample:178 - Decoded Input (Full Prompt + Response)
|||IP_ADDRESS|||<|system|>
You are a helpful math tutor. Solve step by step.
<|user|>
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
<|assistant|>
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72|||IP_ADDRESS|||
2026-01-07 13:34:45.444 | INFO     | moe_reft.sft_dataset:validate_one_sample:182 - Decoded Labels (Assistant Response Only)
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and Ma

In [2]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, List, Optional

import torch
from transformers import PreTrainedTokenizerBase

# Whatever you already use in SFTTransform
CROSS_ENTROPY_IGNORE_INDEX = -100  # or import from your constants


@dataclass
class SFTDataCollator:
    tokenizer: PreTrainedTokenizerBase
    label_pad_token_id: int = CROSS_ENTROPY_IGNORE_INDEX
    pad_to_multiple_of: Optional[int] = None

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        # 1. Separate labels so tokenizer.pad only sees model inputs
        labels_list: List[Any] = [f["labels"] for f in features]
        features_for_pad: List[Dict[str, Any]] = [
            {k: v for k, v in f.items() if k != "labels"} for f in features
        ]

        # 2. Let tokenizer.pad handle input_ids / attention_mask
        batch = self.tokenizer.pad(
            features_for_pad,
            padding=True,                 # pad to max length in this batch
            max_length=None,              # or a fixed max_length if you want
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )

        # 3. Manually pad labels to match seq_len of input_ids
        seq_len: int = batch["input_ids"].size(1)
        padded_labels: List[List[int]] = []

        for lbl in labels_list:
            # convert to python list of ints
            if isinstance(lbl, torch.Tensor):
                lbl_list = lbl.tolist()
            else:
                lbl_list = list(lbl)

            # truncate if somehow longer than seq_len
            if len(lbl_list) > seq_len:
                lbl_list = lbl_list[:seq_len]

            pad_len = seq_len - len(lbl_list)
            if pad_len > 0:
                lbl_list = lbl_list + [self.label_pad_token_id] * pad_len

            padded_labels.append(lbl_list)

        batch["labels"] = torch.tensor(padded_labels, dtype=torch.long)

        return batch


In [3]:
from torch.utils.data import DataLoader

collator = SFTDataCollator(tokenizer=train_dataset.tokenizer)

train_loader = DataLoader(
        train_dataset,
        batch_size=32,
        shuffle=False,
        num_workers=1,
        pin_memory=True,
        collate_fn = collator
    )

In [10]:
# for td in train_loader:
#     print(td['input_ids'].shape)
    # break

In [5]:
td['input_ids'].shape

torch.Size([32, 376])

In [ ]:
from moe_reft import sft_dataset

tokenizer = AutoTokenizer.from_pretrained(tokenizer_model_name)
response_template = sft_dataset.extract_response_template(tokenizer)

train_dataset = sft_dataset.SFTDataset(
        source="openai/gsm8k",
        tokenizer=tokenizer,
        response_template_ids=tokenizer(response_template)['input_ids'],
        system_key=None,
        system_message="You are a helpful math tutor. Solve step by step.",
        user_key="question",
        assistant_key="answer",
        split="train",
        name="main",
)

/home/recoverx/astarag/MoE-ReFT/.venv/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [4]:
tokenizer(response_template)['input_ids']

[187, 29, 93, 515, 5567, 49651, 187]

In [1]:
from moe_reft.olmoe import modeling_olmoe
import torch

model = torch.load("gsm8k_run2/checkpoint-epoch0.pt", weights_only=True)

/home/recoverx/astarag/MoE-ReFT/.venv/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [1]:
from moe_reft.olmoe import load_weights
import torch

model, report = load_weights.load_pretrained_with_interventions_from_checkpoint(
        pt_file="math_gsm8k/checkpoint-epoch0.pt",
        hf_model_name_or_path="allenai/OLMoE-1B-7B-0125",
        intervention_patterns=["*.pre_moe_intervention.*", "*.after_moe_intervention.*"],
        map_dtype=torch.float32,  # or choose appropriate dtype
        map_device=torch.device("cuda"),  # or choose appropriate device
        trust_remote_code=False,
        full_parameter_finetuning=False,
    )
print(report.summary())

/home/recoverx/astarag/MoE-ReFT/.venv/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
/home/recoverx/astarag/MoE-ReFT/moe_reft/olmoe/load_weights.py:237: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

Extracting intervention weights from checkpoint: 100%|██████████| 48/48 [00:00<00:00, 2054.21it/s]
2026-01-16 16:06:45.847 | INFO     | moe_reft.olmoe.load_weights:load_pretrained_with_interventions_from_checkpoint:363 - Parameter stats — total: 6927552512, trainable: 8390656
2026-01-16 16:06:45.850 | INFO     | moe_reft.olmoe.load_weights:load_pretrained_with_interventions_from_checkpoint:364 - Loaded 3219 base weights from HF, 48 intervention weights from checkpoint


Total parameters:     6927552512
Trainable parameters: 8390656
Base model weights loaded:  3219
Intervention weights loaded: 48
Copied: 3267 | Skipped (shape): 0 | Skipped (missing): 80 | Skipped (intervention): 0


In [3]:
from moe_reft import sft_dataset
from transformers import AutoTokenizer


tokenizer_model_name = "allenai/OLMoE-1B-7B-0125-Instruct"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_model_name)
response_template = sft_dataset.extract_response_template(tokenizer)
response_template_ids = tokenizer(response_template)["input_ids"]
message_extractor = sft_dataset.KeyBasedMessageExtractor(
        user_key="question",
        assistant_key="answer",
        system_message="You are a helpful math tutor. Solve step by step.",
    )

val_dataset = sft_dataset.SFTDataset(
    source="openai/gsm8k",
    tokenizer=tokenizer,
    response_template_ids=response_template_ids,
    message_extractor=message_extractor,
    split="test",
    name="main",
)

sample = val_dataset[100]

# Construct messages with only system and user turns (no assistant)
messages = [
    {"role": "system", "content": sample["system"] if "system" in sample else "You are a helpful math tutor. Solve step by step."},
    # {"role": "user", "content": "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?"},
    {"role": "user", "content": "Jerome had 4 friends who came to visit him on a certain day. The first friend pressed on the doorbell 20 times before Jerome opened, the second friend pressed on the doorbell 1/4 times more than Jerome's first friend. The third friend pressed on the doorbell 10 times more than the fourth friend. If the fourth friend pressed on the doorbell 60 times, how many doorbell rings did the doorbell make?"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to(model.device)


2026-01-16 16:12:06.368 | INFO     | moe_reft.sft_dataset:validate_one_sample:178 - Decoded Input (Full Prompt + Response)
|||IP_ADDRESS|||<|system|>
You are a helpful math tutor. Solve step by step.
<|user|>
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
<|assistant|>
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72|||IP_ADDRESS|||
2026-01-16 16:12:06.370 | INFO     | moe_reft.sft_dataset:validate_one_sample:182 - Decoded Labels (Assistant Response Only)
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72|||IP_ADDRESS|||


In [ ]:
# Stream the response token by token, using KV cache (past_key_values) for speed
input_length = inputs.shape[-1]
generated_ids = inputs
response = ""
max_tokens = 256
stop_sequences = ["<|system|>", "<|user|>", "<|assistant|>"]

model.eval()
with torch.no_grad():
    past_key_values = None
    for i in range(max_tokens):
        if past_key_values is None:
            # First step: feed prompt as normal
            outputs = model(input_ids=generated_ids, use_cache=True)
        else:
            # Only feed the last generated id, use past_key_values for the rest
            outputs = model(input_ids=next_token_id, past_key_values=past_key_values, use_cache=True)

        next_token_logits = outputs.logits[:, -1, :]
        next_token_id = torch.argmax(next_token_logits, dim=-1, keepdim=True)

        past_key_values = outputs.past_key_values  # update for next iteration

        # Append next token to the generated_seq
        generated_ids = torch.cat([generated_ids, next_token_id], dim=-1)

        token_id = next_token_id.item()
        if token_id == tokenizer.eos_token_id or token_id == tokenizer.pad_token_id:
            break

        token_str = tokenizer.decode([token_id], skip_special_tokens=True)
        response += token_str
        if any(response.endswith(seq) for seq in stop_sequences):
            for seq in stop_sequences:
                if response.endswith(seq):
                    response = response[: -len(seq)]
                    break
            break

        print(token_str, end='', flush=True)

print()  # for new line after streaming

The first friend pressed on the doorbell 20 times.
The second friend pressed on the doorbell 1/4 times more than Jerome's first friend, which is 20 + 1/4 * 20 = 20 + 5 = 25 times.
The third friend pressed on the doorbell 10 times more than the fourth friend, which is 60 + 10 = 70 times.
The total number of doorbell rings is 20 + 25 + 70 = 115 times.
<|system


: 

In [1]:
# Here
import torch
from moe_reft import interventions_config, tiny_sft, read_config, datamodels, sft_dataset
from moe_reft.olmoe import modeling_olmoe, configuration_olmoe, load_weights

model_name: str = "allenai/OLMoE-1B-7B-0125"
tokenizer_model_name: str = "allenai/OLMoE-1B-7B-0125-Instruct"
config_path = "moe_reft/configs/olmoe.yaml"

train_config, interventions_config_, olmoe_config = read_config.load_all_configs(config_path)

custom_model = modeling_olmoe.OlmoeForCausalLM(olmoe_config)

intervention_patterns: list[str] = []
if not olmoe_config.full_parameter_finetuning:
    intervention_patterns = interventions_config.INTERVENTION_PATTERNS



/home/recoverx/astarag/MoE-ReFT/.venv/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
report, custom_model = load_weights.load_hf_into_custom_model(
    hf_model_name_or_path=model_name,
    custom_model=custom_model,
    intervention_patterns=intervention_patterns,
    full_parameter_finetuning=olmoe_config.full_parameter_finetuning,
    map_dtype=torch.float32,  # optional casting
    map_device=torch.device("cuda"),  # optional device move
    trust_remote_code=False,
)

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

Building partial state dict: 100%|██████████| 3219/3219 [00:05<00:00, 595.26it/s]
2026-01-16 12:38:04.578 | INFO     | moe_reft.olmoe.load_weights:load_hf_into_custom_model:190 - Parameter stats — total: 6927552512, trainable: 8390656


Total parameters:     6927552512
Trainable parameters: 8390656


In [9]:
# for name, param in custom_model.named_parameters():
#     if load_weights.matches_any(name, interventions_config.INTERVENTION_PATTERNS):
#         print(name)
#         param.requires_grad = True

# # 5) Print parameter stats
# total_params, trainable_params = load_weights.count_parameters(custom_model)
# print(f"Total parameters:     {total_params}")